# Yolo

**YOLO(You Only Look Once)**는 객체 탐지(Object Detection)를 수행하는 실시간 딥러닝 모델로, 이미지나 비디오에서 객체의 **위치**와 **종류**를 동시에 예측할 수 있다. YOLO는 객체 탐지를 단일 신경망 패스로 수행하며, 기존의 슬라이딩 윈도우 방식이나 영역 제안(region proposal) 기반 모델에 비해 빠르고 효율적이다.

- **단일 패스 처리**: 이미지를 한 번만 보고 모든 객체를 탐지한다
- **실시간 처리**: 빠른 속도로 영상 처리가 가능하다
- **통합 네트워크**: 분류와 위치 추정을 동시에 수행한다

**작동 원리:**

**1단계: 이미지 분할**
- 입력 이미지를 그리드(grid)로 나눈다
- 예: 7x7 또는 13x13 그리드로 분할한다

**2단계: 경계 상자 예측**
- 각 그리드 셀이 여러 개의 경계 상자(bounding box)를 예측한다
- 각 상자는 (x, y, width, height, confidence) 정보를 포함한다

**3단계: 클래스 확률 계산**
- 각 그리드 셀이 객체 클래스 확률을 계산한다
- 예: 사람(0.9), 자동차(0.1), 개(0.05) 등

**4단계: 최종 결과 생성**
- 신뢰도가 높은 예측만 선택한다
- 중복된 예측은 NMS(Non-Maximum Suppression)로 제거한다


**YOLO의 주요 활용 분야:**

1. **자율주행**  
   - 차량 주변의 보행자, 차량, 도로 표지판 등을 실시간으로 감지.

2. **보안 및 감시**  
   - CCTV 영상에서 침입자 감지, 이상 행동 탐지.

3. **의료 영상 분석**  
   - X-ray나 MRI 이미지에서 병변 탐지.

4. **리테일 분석**  
   - 매장에서 고객 행동 분석, 상품 식별.

**YOLO와 다른 객체 탐지 모델 비교**

| **특징**               | **YOLO**                          | **Faster R-CNN**               | **SSD(Single Shot Detector)**   |
|------------------------|------------------------------------|---------------------------------|----------------------------------|
| **처리 속도**          | 매우 빠름 (실시간 가능)            | 느림                            | 중간                             |
| **정확도**             | 높음 (특히 최신 버전)             | 매우 높음                       | 높음                             |
| **구조**               | 단일 단계 (End-to-End)            | 두 단계 (Region Proposal + Detection) | 단일 단계                        |
| **복잡도**             | 낮음                              | 높음                            | 낮음                             |

In [ ]:
%pip install ultralytics


In [ ]:
# YOLO 모델 로드
from ultralytics import YOLO

model = YOLO('yolo11n.pt') # 사전학습된 YOLO11n 모델 로드
model

In [ ]:
results = model('https://ultralytics.com/images/bus.jpg')
results[0].show()
results[0].save(filename = 'results/bus_yolo.jpg')

In [ ]:
results[0].boxes

- cls: 각 박스의 클래스 id(정수 라벨). 예: 0=person, 5=bus(보통 COCO 기준)
- conf: 각 박스의 신뢰도 점수(0~1). 높을수록 더 확신.
- data: 박스 정보를 한 번에 담은 텐서. 보통 [x1, y1, x2, y2, conf, cls] 형태로 행이 박스 1개.
- id: **트래킹(track)**을 쓸 때 객체 id가 들어감. 지금은 None(트래킹 안 함).
- is_track: 현재 결과가 트래킹 결과인지 여부. False면 단순 탐지.
- orig_shape: 원본 이미지 크기 (H, W).
- shape: data 텐서의 크기. 예: [5, 6]이면 박스 5개, 각 박스당 값 6개.
- xyxy: 박스 좌표를 (x1, y1, x2, y2)(좌상단/우하단) 픽셀 기준으로 제공.
- xywh: 박스 좌표를 (x_center, y_center, width, height) 픽셀 기준으로 제공.
- xyxyn: xyxy를 정규화(0~1) 한 값(이미지 너비/높이로 나눈 좌표).
- xywhn: xywh를 정규화(0~1) 한 값.

In [ ]:
results = model('https://images-ext-1.discordapp.net/external/B2P9Bzocdhb6a1nruIxl2IMq4hMqWCGopVyelrhlGW8/https/upload.wikimedia.org/wikipedia/commons/7/75/Makati_intersection.jpg?format=webp')
results[0].show()
results[0].save(filename = 'results/intersection.jpg')

In [ ]:
%pip install gdown

In [ ]:
# https://drive.google.com/file/d/1tMZODkkNA-ypSuxmGTZgthTG8b3JgFRD
!gdown 1tMZODkkNA-ypSuxmGTZgthTG8b3JgFRD  

In [ ]:
video_path = 'Night_Day_Chase.mp4'
results = model(video_path, save = True)

In [ ]:
# 웹캠을 이용한 실시간 YOLO 객체탐지
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

# 클래스 라벨별 색상 설정 함수
def get_colors(num_colors):
    np.random.seed(0)
    return [tuple(np.random.randint(0, 255, 3).tolist()) for _ in range(num_colors)]

# 클래스 라벨 및 색상 설정
class_names = model.names # {클래스 id: 클래스명, ...}
num_classes = len(class_names) # 클래스 개수
colors = get_colors(num_classes) # 클래스별 색상
print(class_names)

In [ ]:
# 웹캠 프레임(ndarray)에 탐지 결과를 그려서 반환하는 함수
def detect_objects(image: np.array):
    results = model(image, verbose=False)
    class_names = model.names

    for result in results:
        boxes = result.boxes.xyxy
        confs = result.boxes.conf
        cls = result.boxes.cls

        for box, confidence, class_id in zip(boxes, confs, cls):
            x1, y1, x2, y2 = map(int, box)
            label = class_names[int(class_id)]
            cv2.rectangle(image, (x1,y1), (x2,y2), colors[int(class_id)],2)
            cv2.putText(image, f'{label} {confidence:.2f}', (x1,y1), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
        
    return image


In [ ]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print('카메라 연결 실패')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    result_image = detect_objects(frame)

    cv2.imshow('Frame', np.array(result_image))

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 리소스 해제
cap.release() # 카메라 장치 해제
cv2.destroyAllWindows() # OpenCV 창 닫음
cv2.waitkey(1) # 창이 자동으로 닫히지 않을경우 닫히도록 해주는 코드 추가

: 